# Output format Control and Structured Output

In [2]:
import json
from pydantic import BaseModel
from anthropic import Anthropic

import os
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
model = "claude-sonnet-5-5"

In [4]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [23]:
SYSTEM_PROMPT = """
You are an order status classifier.

Classify the intent and urgency of each customer message. Use customer's own words to judge urgency. Do not assume urgency based on topic alone.

  - "intent" is one of: "status_check", "cancel", "modify_status", "other"
  - "urgency" is one of: "high", "medium", "low\"

"""

class OrderIntent(BaseModel):
    order_id: str | None
    intent: str
    urgency: str

max_tokens = 256

def classify_order_intent(user_message: str) -> OrderIntent:
    response = client.messages.parse(
        model= model,
        max_tokens=max_tokens,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
        output_format=OrderIntent
    )
    return response.parsed_output

In [30]:
print("==== Classifying order intent: JSON output with a pydantic model ====\n")

CLASSIFY_CASES = [
    "I was charged $120, but my plan is only $100. Can you explain the charge?",
    "URGENT: Please cancel my order ORD-123 immediately! I ordered the wrong size.",
]

for case in CLASSIFY_CASES:
    result = classify_order_intent(case)
    print(f"Input: {case[:70]}{'...' if len(case) > 70 else ''}")
    print(f"Output: {result.model_dump_json()}\n")
    print()

==== Classifying order intent: JSON output with a pydantic model ====

Input: I was charged $120, but my plan is only $100. Can you explain the char...
Output: {"order_id":null,"intent":"other","urgency":"medium"}


Input: URGENT: Please cancel my order ORD-123 immediately! I ordered the wron...
Output: {"order_id":"ORD-123","intent":"cancel","urgency":"high"}




In [28]:
EXTRACT_SYSTEM = """
Extract invoice fields from the text.
Use vendor's own words for line items; do not summarize or abbreviate. 
"""

class Invoice(BaseModel):
    vendor_name: str
    invoice_date: str
    line_items: list[str] 
    total_amount: str

max_tokens = 512

def extract_invoice(text: str) -> Invoice:
    response = client.messages.parse(
        model=model,
        max_tokens=max_tokens,
        system=EXTRACT_SYSTEM,
        messages=[{"role": "user", "content": text}],
        output_format=Invoice
    )
    return response.parsed_output

In [29]:
print("==== Extract invoice: JSON output with a pydantic model ====\n")

invoice = (
    "INVOICE\n"
    "Vendor: Acme Supplies\n"
    "Invoice Date: 2024-06-01\n"
    "Line Items:\n"
    "  - Item: Widget A, Quantity: 10, Price: $5.00\n"
    "  - Item: Widget B, Quantity: 5, Price: $45.00\n"
    "Total Amount: $275.00\n"   
)

result = extract_invoice(invoice)
print(result.model_dump_json(indent=2))
print() 

==== Extract invoice: JSON output with a pydantic model ====

{
  "vendor_name": "Acme Supplies",
  "invoice_date": "2024-06-01",
  "line_items": [
    "Item: Widget A, Quantity: 10, Price: $5.00",
    "Item: Widget B, Quantity: 5, Price: $45.00"
  ],
  "total_amount": "$275.00"
}



In [31]:
FLIGHT_TOOL = {
    "name": "search_flights",
    "strict": True,
    "input_schema": {
        "type": "object", 
        "properties": {
            "destination": {"type": "string"},
            "date": {"type": "string", "format": "date"}
        },
        "required": ["destination", "date"],
        "additionalProperties": False
    }
}

TRIP_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "next_steps": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "next_steps"],
    "additionalProperties": False
}

max_tokens = 512
def plan_trip(user_message: str) -> dict:
    response = client.messages.parse(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": user_message}],
        tools=[FLIGHT_TOOL],
        output_config = {"format": {"type": "json_schema", "schema": TRIP_SCHEMA}}
    )
    tool_call = next((msg for msg in response.content if msg.type == "tool_use"), None)
    text_block = next((msg for msg in response.content if msg.type == "text"), None)
    return {
        "tool_call": tool_call,
        "text_block": text_block    
    }

In [35]:
print("=== Plan trip: JSON output combines with strict tool use ===\n")
result = plan_trip("Help me plan a trip to Paris departing November 17, 2026")

print(result)

=== Plan trip: JSON output combines with strict tool use ===

{'tool_call': ToolUseBlock(id='toolu_017HBSKNPWRVAirdeHKRVb4X', caller=DirectCaller(type='direct'), input={'destination': 'Paris', 'date': '2026-11-17'}, name='search_flights', type='tool_use', toolset_name=None), 'text_block': None}


In [38]:
def json_default(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    raise TypeError(f"{type(obj).__name__} is not JSON serializable")

print(json.dumps(result, default=json_default, indent=2))

{
  "tool_call": {
    "id": "toolu_017HBSKNPWRVAirdeHKRVb4X",
    "caller": {
      "type": "direct"
    },
    "input": {
      "destination": "Paris",
      "date": "2026-11-17"
    },
    "name": "search_flights",
    "type": "tool_use",
    "toolset_name": null
  },
  "text_block": null
}
